In [5]:
!pip install -U langchain-openai

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 1.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 3.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 3.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.5/801.5 kB 3.6 MB/s eta 0:00:0000:0100:01
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)


In [ ]:
import os
import langchain
import pydantic
from enum import Enum
from pydantic import BaseModel, Field

from dotenv import load_dotenv
import os
from langchain_core.prompts import ChatPromptTemplate

In [2]:
class RequestType(str, Enum):
    QUESTION = "question"
    TASK = "task"
    SMALL_TALK = "small_talk"
    COMPLAINT = "complaint"
    UNKNOWN = "unknown"

class Classification(BaseModel):
    request_type: RequestType = Field(description="Тип запроса")
    confidence: float = Field(ge=0, le=1, description="Уверенность от 0 до 1")
    reasoning: str = Field(description="Краткое обоснование")

class AssistantResponse(BaseModel):
    content: str = Field(description="Текст ответа")
    request_type: RequestType = Field(description="Тип запроса")
    confidence: float = Field(ge=0, le=1, description="Уверенность от 0 до 1")
    tokens_used: int = Field(description="Приблизительное число токенов")



In [6]:
load_dotenv()


True

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

model = ChatOpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free",
    temperature=0,
)

parser = PydanticOutputParser(pydantic_object=Classification)

classification_prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты — классификатор пользовательских запросов.

Определи тип запроса.

Типы:
- question — вопрос, требующий информации («Что такое Python?», «Как работает GIL?»)
- task — просьба что-то сделать («Напиши стих», «Расскажи анекдот»)
- small_talk — приветствие и болтовня («Привет!», «Как дела?»)
- complaint — жалоба, недовольство («Это ужасно работает!», «Почему так долго?»)
- unknown — бессмыслица или нераспознанный запрос («asdfghjkl»)

Примеры:
Запрос: Привет!
Ответ: small_talk

Запрос: Как дела?
Ответ: small_talk

Запрос: Что такое Python?
Ответ: question

Запрос: Что такое LangChain?
Ответ: question

Запрос: Напиши стих про осень
Ответ: task

Запрос: Расскажи шутку про ёжика
Ответ: task

Запрос: Почему всё так долго работает?!
Ответ: complaint

Запрос: Почему твой код не работает?
Ответ: complaint

Запрос: asdfghjkl
Ответ: unknown

Запрос: &ghh&
Ответ: unknown

Верни результат строго в формате:
{format_instructions}
"""),
    ("human", "Запрос: {query}")
])

chain = (
    {
        "query": RunnablePassthrough(),
        "format_instructions": lambda _: parser.get_format_instructions(),
    }
    | classification_prompt
    | model
    | parser
)

try:
    result = chain.invoke("Что такое dict?")
except Exception:
    result = Classification(
        request_type=RequestType.UNKNOWN,
        confidence=0.5,
        reasoning="Ошибка парсинга ответа модели"
    )

print(f"request_type: {result.request_type.value}")
print(f"confidence: {result.confidence}")
print(f"reasoning: {result.reasoning}")

request_type: question
confidence: 0.95
reasoning: Запрос 'Что такое dict?' явно задает вопрос о значении или природе объекта 'dict', что является типичным для вопроса, требующего информации.


In [22]:
classification = chain.invoke("Что такое dict?")
print(classification.request_type)

RequestType.QUESTION


In [23]:
from langchain_core.output_parsers import StrOutputParser

TYPE_PROMPTS = {
    RequestType.QUESTION: "Дай информативный и полезный ответ. Если не знаешь — честно скажи.",
    RequestType.TASK: "Пользователь просит выполнить задачу. Сделай это качественно.",
    RequestType.SMALL_TALK: "Поддержи беседу, будь приветлив.",
    RequestType.COMPLAINT: "Прояви эмпатию, постарайся понять проблему и предложить решение.",
    RequestType.UNKNOWN: "Запрос непонятен. Вежливо попроси пользователя уточнить, что он имел в виду."
}

def build_handler(system_prompt: str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{query}")
    ])
    return prompt | model | StrOutputParser()

handlers = {
    request_type: build_handler(system_prompt)
    for request_type, system_prompt in TYPE_PROMPTS.items()
}

In [ ]:
def process_query(query: str) -> AssistantResponse:
    # 1. Классификация
    try:
        classification = chain.invoke(query)
    except Exception:
        classification = Classification(
            request_type=RequestType.UNKNOWN,
            confidence=0.5,
            reasoning="Ошибка парсинга ответа модели"
        )

    handler = handlers.get(classification.request_type, handlers[RequestType.UNKNOWN])

    try:
        content = handler.invoke({"query": query})
    except Exception:
        content = "Не удалось сгенерировать ответ. Попробуйте ещё раз."

    return AssistantResponse(
        content=content,
        request_type=classification.request_type,
        confidence=classification.confidence,
        tokens_used=len(query.split()) + len(content.split())  # пока грубая оценка
    )

In [25]:
test_queries = [
    "Что такое Python?",
    "Напиши стих про весну",
    "Привет, как дела?",
    "Это ужасно работает!",
    "asdfghjkl"
]

for q in test_queries:
    response = process_query(q)
    print(f"Запрос: {q}")
    print(f"[{response.request_type.value}] {response.content}")
    print(f"confidence: {response.confidence} | tokens: ~{response.tokens_used}")
    print("-" * 80)

Запрос: Что такое Python?
[question] Python — это язык программирования, очень простой и универсал, который используется в разных областях, таких как разработка программ, аналитика, ciencia data, а также в игровой индустрии. Он известен своей читаемой синтаксисом, что делает его удобным для начинающих. Если нужно, я могу объяснить его более подробно!
confidence: 0.95 | tokens: ~48
--------------------------------------------------------------------------------
Запрос: Напиши стих про весну
[task] 
confidence: 1.0 | tokens: ~4
--------------------------------------------------------------------------------
Запрос: Привет, как дела?
[small_talk] Привет! У меня всё отлично, спасибо за вопрос. А у вас как дела?
confidence: 0.95 | tokens: ~16
--------------------------------------------------------------------------------
Запрос: Это ужасно работает!
[complaint] Я искренне сожалею, что вы так расстроены! Могу ли я узнать, о чём именно вы жалуетесь? Чем "ужасно" работает — это продукт, услуг

In [26]:
from langchain_core.output_parsers import StrOutputParser

CHARACTER_PROMPTS = {
    "friendly": (
        "Ты дружелюбный и позитивный ассистент. "
        "Отвечай тепло, понятно и естественно. "
        "Можно иногда использовать эмодзи, но умеренно."
    ),
    "professional": (
        "Ты профессиональный и сдержанный ассистент. "
        "Отвечай чётко, вежливо и по делу, без лишней фамильярности."
    ),
    "sarcastic": (
        "Ты ассистент с лёгкой иронией. "
        "Отвечай с юмором, но не груби и обязательно оставайся полезным."
    ),
    "pirate": (
        "Ты ассистент-пират. "
        "Говори как пират: можно использовать 'Арр', 'матрос', "
        "'тысяча чертей', но ответ всё равно должен быть понятным и полезным."
    ),
}

TYPE_PROMPTS = {
    RequestType.QUESTION: (
        "Это вопрос пользователя. "
        "Дай информативный и полезный ответ. "
        "Если не знаешь — честно скажи об этом."
    ),
    RequestType.TASK: (
        "Пользователь просит выполнить задачу. "
        "Сделай это качественно и понятно."
    ),
    RequestType.SMALL_TALK: (
        "Это дружеское общение. "
        "Поддержи беседу, будь приветлив. "
        "Если пользователь представился, отреагируй естественно."
    ),
    RequestType.COMPLAINT: (
        "Пользователь жалуется или недоволен. "
        "Прояви эмпатию, постарайся понять проблему и предложи решение."
    ),
    RequestType.UNKNOWN: (
        "Запрос непонятен. "
        "Вежливо попроси пользователя уточнить, что он имел в виду."
    ),
}

def build_handler(request_type: RequestType, character: str):
    system_prompt = (
        f"{CHARACTER_PROMPTS[character]}\n\n"
        f"{TYPE_PROMPTS[request_type]}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{query}")
    ])

    return prompt | model | StrOutputParser()


def create_handlers(character: str):
    return {
        request_type: build_handler(request_type, character)
        for request_type in RequestType
    }

In [ ]:
current_character = "friendly"
handlers = create_handlers(current_character)

def set_character(new_character: str):
    global current_character, handlers

    if new_character not in CHARACTER_PROMPTS:
        available = ", ".join(CHARACTER_PROMPTS.keys())
        raise ValueError(f"Неизвестный характер: {new_character}. Доступно: {available}")

    current_character = new_character
    handlers = create_handlers(current_character)
    return f"Характер изменён на: {current_character}"


def process_query(query: str) -> AssistantResponse:
    try:
        classification = chain.invoke(query)
    except Exception:
        classification = Classification(
            request_type=RequestType.UNKNOWN,
            confidence=0.5,
            reasoning="Ошибка парсинга ответа модели"
        )

    handler = handlers.get(
        classification.request_type,
        handlers[RequestType.UNKNOWN]
    )

    try:
        content = handler.invoke({"query": query})
    except Exception:
        content = "Не удалось сгенерировать ответ. Попробуйте ещё раз."

    tokens_used = len(query.split()) + len(content.split())

    return AssistantResponse(
        content=content,
        request_type=classification.request_type,
        confidence=classification.confidence,
        tokens_used=tokens_used
    )

In [28]:
print(set_character("sarcastic"))
response = process_query("Что такое Python?")
print(f"[{response.request_type.value}] {response.content}")
print(f"confidence: {response.confidence} | tokens: ~{response.tokens_used}")

Характер изменён на: sarcastic
[question] **Python**— это язык программирования, который в 1991‑м году решил, что «всё, что можно написать на английском, тоже можно написать на «змеином»».  

- **Синтаксис** прост, почти как английский: `print("Привет, мир!")` выводит приветствие, а `for i in range(10):` перебирает числа от 0 до 9.  
- **Библиотеки** — как набор волшебных палочек: `numpy` для чисел, `pandas` для таблиц, `requests` для общения с интернетом и даже `pygame` для простых игр.  
- **Платформенная независимость**: пишешь один раз — работает везде (Windows, macOS, Linux, даже на Raspberry Pi).  
- **Сообщество** — огромный набор «змейко‑энтузиастов», которые делятся готовыми решениями, а ты можешь просто скопировать их и выглядеть как профи.  

**Итог:** Python — это «язык, в котором ты можешь писать код, не разрываясь от мысли», а также отличный способ начать программировать без головной боли. Если хочешь попробовать, просто скачай официальную версию с python.org и напиши пер

In [29]:
print(set_character("pirate"))
response = process_query("Привет!")
print(f"[{response.request_type.value}] {response.content}")
print(f"confidence: {response.confidence} | tokens: ~{response.tokens_used}")

Характер изменён на: pirate
[small_talk] Арр, матрос! Ты здесь, на борту моего корабля? Говори, что тебя привело на мои земли? Нужна помощь в поиске сокровищ или, может, просто хочешь поболтать с капитаном?

confidence: 1.0 | tokens: ~28


In [30]:
from typing import List, Optional
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.prompts import MessagesPlaceholder

class MemoryManager:
    def __init__(
        self,
        model,
        strategy: str = "buffer",
        max_messages: int = 20,
        keep_last: int = 6,
    ):
        self.model = model
        self.strategy = strategy
        self.max_messages = max_messages
        self.keep_last = keep_last
        self.messages: List[BaseMessage] = []
        self.summary: Optional[str] = None

    def add_user_message(self, text: str):
        self.messages.append(HumanMessage(content=text))
        self._trim_or_summarize()

    def add_ai_message(self, text: str):
        self.messages.append(AIMessage(content=text))
        self._trim_or_summarize()

    def clear(self):
        self.messages = []
        self.summary = None

    def set_strategy(self, strategy: str):
        if strategy not in ["buffer", "summary"]:
            raise ValueError("Стратегия памяти должна быть 'buffer' или 'summary'")
        self.strategy = strategy

    def get_history(self) -> List[BaseMessage]:
        if self.strategy == "summary" and self.summary:
            return [SystemMessage(content=f"Краткое содержание предыдущего диалога: {self.summary}")] + self.messages
        return self.messages

    def message_count(self) -> int:
        return len(self.messages)

    def _trim_or_summarize(self):
        if self.strategy == "buffer":
            if len(self.messages) > self.max_messages:
                self.messages = self.messages[-self.max_messages:]
            return

        if self.strategy == "summary" and len(self.messages) > self.max_messages:
            old_messages = self.messages[:-self.keep_last]
            recent_messages = self.messages[-self.keep_last:]

            dialogue_text = "\n".join(
                f"{msg.__class__.__name__}: {msg.content}" for msg in old_messages
            )

            summary_prompt = ChatPromptTemplate.from_messages([
                ("system", "Сделай краткое и точное summary диалога. Сохрани факты о пользователе, его предпочтения, цели и важный контекст."),
                ("human", "Текущий summary:\n{current_summary}\n\nДиалог для сжатия:\n{dialogue_text}")
            ])

            summary_chain = summary_prompt | model | StrOutputParser()

            try:
                self.summary = summary_chain.invoke({
                    "current_summary": self.summary or "Пока summary нет.",
                    "dialogue_text": dialogue_text
                })
            except Exception:
                pass

            self.messages = recent_messages

In [31]:
def build_handler(request_type: RequestType, character: str):
    system_prompt = (
        f"{CHARACTER_PROMPTS[character]}\n\n"
        f"{TYPE_PROMPTS[request_type]}"
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{query}")
    ])

    return prompt | model | StrOutputParser()

In [32]:
current_character = "friendly"
handlers = create_handlers(current_character)
memory = MemoryManager(model=model, strategy="buffer", max_messages=20, keep_last=6)

In [33]:
def process_query(query: str) -> AssistantResponse:
    try:
        classification = chain.invoke(query)
    except Exception:
        classification = Classification(
            request_type=RequestType.UNKNOWN,
            confidence=0.5,
            reasoning="Ошибка парсинга ответа модели"
        )

    handler = handlers.get(
        classification.request_type,
        handlers[RequestType.UNKNOWN]
    )

    history = memory.get_history()

    try:
        content = handler.invoke({
            "query": query,
            "history": history
        })
    except Exception:
        content = "Не удалось сгенерировать ответ. Попробуйте ещё раз."

    memory.add_user_message(query)
    memory.add_ai_message(content)

    tokens_used = len(query.split()) + len(content.split())

    return AssistantResponse(
        content=content,
        request_type=classification.request_type,
        confidence=classification.confidence,
        tokens_used=tokens_used
    )

In [34]:
def set_memory_strategy(strategy: str):
    memory.set_strategy(strategy)
    return f"Стратегия памяти изменена на: {strategy}"

def clear_memory():
    memory.clear()
    return "История диалога очищена."

In [35]:
print(clear_memory())
print(set_memory_strategy("buffer"))

r1 = process_query("Привет, меня зовут Даша")
print(f"[{r1.request_type.value}] {r1.content}")

r2 = process_query("Мой любимый язык — Python")
print(f"[{r2.request_type.value}] {r2.content}")

r3 = process_query("Как меня зовут и какой мой любимый язык?")
print(f"[{r3.request_type.value}] {r3.content}")

История диалога очищена.
Стратегия памяти изменена на: buffer
[small_talk] Привет, Даша! Очень приятно познакомиться 🌟 Как твои дела?
[small_talk] Python — отличный выбор! Это очень мощный и в то же время простой язык программирования. Чем ты любишь заниматься на Python? Может быть, пишешь какие-то интересные проекты или изучаешь какие-то конкретные области, например, машинное обучение или веб-разработку?
[question] Тебя зовут Даша, атвой любимый язык программирования — Python! 😊 Отличный выбор — он действительно универсален и приятен в работе. Чем ты сейчас занимаешься на Python? Может, есть интересный проект, которым хочешь поделиться?


In [41]:
import argparse

class SmartAssistant:
    def __init__(self, model_name: str = "gpt-4o-mini", character: str = "friendly", memory_strategy: str = "buffer"):
        self.model_name = model_name
        self.character = character
        self.memory_strategy = memory_strategy

        # Используем уже созданные глобальные model / chain / функции
        set_character(character)
        set_memory_strategy(memory_strategy)

    def process(self, query: str) -> AssistantResponse:
        return process_query(query)

    def set_character(self, character: str):
        set_character(character)
        self.character = character

    def set_memory(self, strategy: str):
        set_memory_strategy(strategy)
        self.memory_strategy = strategy

    def clear(self):
        clear_memory()

    def status(self) -> str:
        return (
            f"Характер: {self.character} | "
            f"Память: {self.memory_strategy} | "
            f"Сообщений в истории: {memory.message_count()}"
        )

In [42]:
def print_help():
    print("""
Доступные команды:
/help                      — показать справку
/status                    — показать текущие настройки
/clear                     — очистить историю диалога
/character <name>          — сменить характер (friendly, professional, sarcastic, pirate)
/memory <strategy>         — сменить память (buffer, summary)
/quit                      — выйти
""".strip())


def handle_command(user_input: str, assistant: SmartAssistant) -> bool:
    parts = user_input.strip().split(maxsplit=1)
    command = parts[0].lower()

    if command == "/help":
        print_help()
        return True

    if command == "/status":
        print(assistant.status())
        return True

    if command == "/clear":
        assistant.clear()
        print("✓ История очищена")
        return True

    if command == "/character":
        if len(parts) < 2:
            print("Укажи характер: friendly, professional, sarcastic, pirate")
            return True
        try:
            assistant.set_character(parts[1].strip())
            print(f"✓ Характер изменён на: {assistant.character}")
        except Exception as e:
            print(f"Ошибка: {e}")
        return True

    if command == "/memory":
        if len(parts) < 2:
            print("Укажи стратегию памяти: buffer или summary")
            return True
        try:
            assistant.set_memory(parts[1].strip())
            print(f"✓ Память изменена на: {assistant.memory_strategy}")
        except Exception as e:
            print(f"Ошибка: {e}")
        return True

    if command == "/quit":
        print("Пока!")
        return False

    print("Неизвестная команда. Введи /help")
    return True

In [43]:
def run_cli(character: str = "friendly", memory_strategy: str = "buffer", model_name: str = "gpt-4o-mini"):
    assistant = SmartAssistant(
        model_name=model_name,
        character=character,
        memory_strategy=memory_strategy
    )

    print("🤖 Умный ассистент с характером")
    print(f"Характер: {assistant.character} | Память: {assistant.memory_strategy}")
    print("────────────────────────────────")

    while True:
        try:
            user_input = input("\n> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nПока!")
            break

        if not user_input:
            continue

        if user_input.startswith("/"):
            should_continue = handle_command(user_input, assistant)
            if not should_continue:
                break
            continue

        response = assistant.process(user_input)
        print(f"[{response.request_type.value}] {response.content}")
        print(f"confidence: {response.confidence} | tokens: ~{response.tokens_used}")

In [44]:
def main():
    parser = argparse.ArgumentParser(description="Умный ассистент с характером")
    parser.add_argument("--character", default="friendly", choices=["friendly", "professional", "sarcastic", "pirate"])
    parser.add_argument("--memory", default="buffer", choices=["buffer", "summary"])
    parser.add_argument("--model", default="gpt-4o-mini")

    args = parser.parse_args()

    run_cli(
        character=args.character,
        memory_strategy=args.memory,
        model_name=args.model
    )

In [45]:
if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h]
                             [--character {friendly,professional,sarcastic,pirate}]
                             [--memory {buffer,summary}] [--model MODEL]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/1000/jupyter/runtime/kernel-v31f154965bfa79488d6905f9fd569590542a190b0.json


SystemExit: 2

In [47]:
run_cli(character="friendly", memory_strategy="buffer", model_name="gpt-4o-mini")

🤖 Умный ассистент с характером
Характер: friendly | Память: buffer
────────────────────────────────
[small_talk] У меня всё отлично, спасибо! 😊 А как у тебя дела, Даша? Как проходит день?
confidence: 0.95 | tokens: ~17
[small_talk] Привет, Ксюша! Ксюша — это очень приятное имя! 🍎 Ты действительно любешь яблоки — это очень хороший выбор! Я обожаю это. Как дела с твоими яблок? Или ты делаешь что-то особенное с ними?
confidence: 0.95 | tokens: ~40
[question] Привет, Ксюша! 🍎 Я счастлива, что ты упомянул о том, что ты любит яблоки — это так приятно! Я тоже ценил такую позитивную энергию. Как ты сегодня? Надеюсь, тоже тоже наслаждаешься чем-то особенным! 😊 Чем ты готовить или что делаешь в дне? С удовольствием расскажу больше!
confidence: 1.0 | tokens: ~49

Пока!
